# EcoHome Energy Advisor - RAG Setup

Retrieval-Augmented Generation over the energy-saving knowledge base.

Documents in `data/documents/` include the two starter files plus HVAC strategies, smart-home automation, renewable integration, seasonal management, and energy storage.


## 1. Import Required Libraries


In [1]:
import os, sys
from pathlib import Path

ROOT = Path("/Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd:", Path.cwd())
print("models exists:", (ROOT / "models" / "energy.py").exists())

cwd: /Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution
models exists: True


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()


True

## 2. Load and Process Documents


In [3]:
documents = []
document_paths = sorted(str(p) for p in Path('data/documents').glob('*.txt'))
print('Knowledge-base files:')
for p in document_paths:
    print(' -', p)
    text = Path(p).read_text(encoding='utf-8')
    documents.append(type('Doc', (), {'page_content': text, 'metadata': {'source': p}})())
print(f'Total documents loaded: {len(documents)}')



Knowledge-base files:
 - data/documents/tip_device_best_practices.txt
 - data/documents/tip_energy_savings.txt
 - data/documents/tip_energy_storage_optimization.txt
 - data/documents/tip_ev_charging_strategies.txt
 - data/documents/tip_hvac_optimization.txt
 - data/documents/tip_hvac_optimization_strategies.txt
 - data/documents/tip_outage_and_backup_planning.txt
 - data/documents/tip_pool_pump_best_practices.txt
 - data/documents/tip_renewable_energy_integration.txt
 - data/documents/tip_seasonal_energy_management.txt
 - data/documents/tip_smart_home_automation.txt
 - data/documents/tip_solar_panel_efficiency.txt
 - data/documents/tip_time_of_use_planning.txt
Total documents loaded: 13


## 3. Split Documents into Chunks


In [4]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_core.documents import Document
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents([Document(page_content=d.page_content, metadata=d.metadata) for d in documents])
except Exception:
    splits = []
    for d in documents:
        for block in d.page_content.split('\n\n'):
            if block.strip():
                splits.append(type('C', (), {'page_content': block.strip(), 'metadata': d.metadata})())
print(f'Split {len(documents)} documents into {len(splits)} chunks')
if splits:
    print('Sample chunk:', splits[0].page_content[:200], '...')



Split 13 documents into 29 chunks
Sample chunk: Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during ...


## 4. Create Vector Store


In [5]:
persist_directory = 'data/vectorstore'
os.makedirs(persist_directory, exist_ok=True)
api_key = os.getenv('VOCAREUM_API_KEY') or os.getenv('OPENAI_API_KEY')
vectorstore = None
if api_key:
    try:
        from langchain_chroma import Chroma
        from langchain_openai import OpenAIEmbeddings
        from langchain_core.documents import Document
        kwargs = {'api_key': api_key}
        if os.getenv('VOCAREUM_API_KEY'):
            kwargs['base_url'] = os.getenv('OPENAI_BASE_URL', 'https://openai.vocareum.com/v1')
        embeddings = OpenAIEmbeddings(**kwargs)
        docs = [Document(page_content=s.page_content, metadata=s.metadata) for s in splits]
        vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=persist_directory)
        print(f'Vector store created at {persist_directory} with {len(docs)} vectors')
    except Exception as exc:
        print('Chroma/OpenAI unavailable:', exc)
else:
    print('No API key — search_energy_tips will use the local TF-IDF / lexical retriever.')



Vector store created at data/vectorstore with 29 vectors


## 5. Test the RAG Pipeline


In [6]:
from tools import search_energy_tips
test_queries = [
    'electric vehicle charging tips',
    'thermostat optimization',
    'dishwasher energy saving',
    'solar power maximization',
    'HVAC system efficiency',
    'pool pump scheduling',
    'home battery dispatch',
    'monsoon seasonal solar',
]
print('=== Testing search_energy_tips ===')
for query in test_queries:
    result = search_energy_tips.invoke({'query': query, 'max_results': 2})
    print(f"\nQuery: {query}")
    print(f"  results={result.get('total_results')} retriever={result.get('retriever', 'chroma')}")
    for tip in result.get('tips', [])[:2]:
        print(f"    {tip['rank']}. ({tip['source']}) {tip['content'][:90].replace(chr(10),' ')}...")



=== Testing search_energy_tips ===

Query: electric vehicle charging tips
  results=2 retriever=chroma
    1. (data/documents/tip_device_best_practices.txt) Large devices like electric vehicles, washing machines and dishwashers often support delay...
    2. (data/documents/tip_device_best_practices.txt) Large devices like electric vehicles, washing machines and dishwashers often support delay...

Query: thermostat optimization
  results=2 retriever=chroma
    1. (data/documents/tip_hvac_optimization.txt) HVAC and Thermostat Optimization ================================  In a warm climate the a...
    2. (data/documents/tip_hvac_optimization.txt) HVAC and Thermostat Optimization ================================  In a warm climate the a...

Query: dishwasher energy saving
  results=2 retriever=chroma
    1. (data/documents/tip_device_best_practices.txt) Dishwasher Best Practices: - Only run when completely full - Use the energy-saving or eco ...
    2. (data/documents/tip_device_best_pra